# Feature Engineering

Input: `science_sample_balanced.pkl` (raw balanced sample of science vs. general subreddit posts)

Output: `data_final_features.pkl` (cleaned dataset with all engineered features: semantic similarity, readability, compression ratio, profanity, sentiment, subjectivity, entity retention, pronoun rate)

In [ ]:
# Install / verify all required packages
%pip install -q sentence-transformers textstat better-profanity vaderSentiment textblob spacy seaborn scikit-learn statsmodels
!python -m spacy download en_core_web_sm --quiet
print("All packages ready.")

In [ ]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import textstat
from better_profanity import profanity
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import spacy

## Load Data

In [ ]:
data_final = pd.read_pickle("science_sample_balanced.pkl")
print(data_final.shape)
data_final.head()

## Semantic Similarity (SBERT Cosine Similarity)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
content_embeddings = model.encode(
    data_final["content"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

summary_embeddings = model.encode(
    data_final["summary"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

eps = 1e-8  # avoid division by zero

data_final["cosine_similarity"] = (
    np.sum(content_embeddings * summary_embeddings, axis=1)
    / (
        np.linalg.norm(content_embeddings, axis=1)
        * np.linalg.norm(summary_embeddings, axis=1)
        + eps
    )
)

data_final["cosine_similarity"].describe()

## Readability Metric

In [ ]:
def compute_readability(text):
    """Compute multiple readability scores for a given text."""
    return {
        'flesch_reading_ease': textstat.flesch_reading_ease(text),
        'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
        'gunning_fog': textstat.gunning_fog(text),
        'coleman_liau': textstat.coleman_liau_index(text)
    }

content_readability = data_final['content'].apply(compute_readability)
summary_readability = data_final['summary'].apply(compute_readability)

content_readability_df = pd.DataFrame(content_readability.tolist()).add_prefix('content_')
summary_readability_df = pd.DataFrame(summary_readability.tolist()).add_prefix('summary_')

data_final = pd.concat([data_final, content_readability_df, summary_readability_df], axis=1)

readability_metrics = ['flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog', 'coleman_liau']
for metric in readability_metrics:
    data_final[f'delta_{metric}'] = (
        data_final[f'summary_{metric}'] - data_final[f'content_{metric}']
    )

## Cleaning Data

In [ ]:
n_before = len(data_final)

# 1. Remove duplicates
data_final = data_final.drop_duplicates(subset='content')

# 2. Filter implausible readability scores on content side (lists, markup artifacts)
data_final = data_final[data_final['content_flesch_kincaid_grade'] < 30]

# 3. Filter summaries that are too short to be meaningful
data_final = data_final[data_final['summary_len'] > 3]

n_after = len(data_final)
print(f"Removed {n_before - n_after} observations ({(n_before - n_after) / n_before * 100:.1f}%)")
print(f"Remaining: {n_after}")

In [ ]:
# Non-alphabetic character ratio (lists, markup artifacts, etc.)
data_final['content_alpha_ratio'] = data_final['content'].apply(
    lambda x: sum(c.isalpha() for c in x) / len(x) if len(x) > 0 else 0
)
data_final = data_final[data_final['content_alpha_ratio'] >= 0.5]
print(f"Remaining: {len(data_final)}")

In [ ]:
print(data_final['is_science'].value_counts())
print(data_final['is_science'].value_counts(normalize=True).round(3))

## Compression Ratio

In [ ]:
data_final['compression_ratio'] = data_final['summary_len'] / data_final['content_len']
data_final['compression_ratio'].describe()

## Profanity Metric

In [ ]:
data_final['content'] = data_final['content'].astype(str)
data_final['summary'] = data_final['summary'].astype(str)

clean_wordset = [str(w) for w in profanity.CENSOR_WORDSET]
profanity_pattern = r'\b(' + '|'.join(re.escape(w) for w in clean_wordset) + r')\b'

data_final['content_profanity'] = data_final['content'].str.lower().str.contains(
    profanity_pattern, regex=True, na=False).astype(int)
data_final['summary_profanity'] = data_final['summary'].str.lower().str.contains(
    profanity_pattern, regex=True, na=False).astype(int)

data_final['delta_profanity'] = data_final['summary_profanity'] - data_final['content_profanity']

## Sentiment and Subjectivity

In [ ]:
analyzer = SentimentIntensityAnalyzer()

data_final['content_sentiment'] = data_final['content'].apply(
    lambda x: analyzer.polarity_scores(x)['compound'])
data_final['summary_sentiment'] = data_final['summary'].apply(
    lambda x: analyzer.polarity_scores(x)['compound'])

data_final['content_subjectivity'] = data_final['content'].apply(
    lambda x: TextBlob(x).sentiment.subjectivity)
data_final['summary_subjectivity'] = data_final['summary'].apply(
    lambda x: TextBlob(x).sentiment.subjectivity)

data_final['delta_sentiment'] = data_final['summary_sentiment'] - data_final['content_sentiment']
data_final['delta_subjectivity'] = data_final['summary_subjectivity'] - data_final['content_subjectivity']

## Entity Retention Rate

In [ ]:
nlp = spacy.load('en_core_web_sm')

def entity_retention_rate(content, summary):
    content_entities = set(ent.text.lower() for ent in nlp(content).ents)
    summary_entities = set(ent.text.lower() for ent in nlp(summary).ents)

    if len(content_entities) == 0:
        return None

    retained = content_entities.intersection(summary_entities)
    return len(retained) / len(content_entities)

data_final['entity_retention_rate'] = data_final.apply(
    lambda row: entity_retention_rate(row['content'], row['summary']), axis=1)

data_final['entity_retention_rate'] = data_final['entity_retention_rate'].fillna(
    data_final['entity_retention_rate'].median()
)

## Pronoun Usage

In [ ]:
# Pronoun usage as a register feature
# Personal pronouns signal narrative/interpersonal vs. formal/impersonal writing
pronoun_pattern = re.compile(
    r'\b(i|me|my|mine|myself|'
    r'we|us|our|ours|ourselves|'
    r'you|your|yours|yourself|yourselves|'
    r'he|him|his|she|her|hers|'
    r'they|them|their|theirs|themselves|'
    r'himself|herself)\b',
    re.IGNORECASE
)
word_pattern = re.compile(r'\b[a-zA-Z]+\b')

def pronoun_rate(text):
    words = word_pattern.findall(str(text))
    if not words:
        return 0.0
    return len(pronoun_pattern.findall(str(text))) / len(words)

data_final['content_pronoun_rate'] = data_final['content'].apply(pronoun_rate)
data_final['summary_pronoun_rate'] = data_final['summary'].apply(pronoun_rate)
data_final['delta_pronoun_rate'] = data_final['summary_pronoun_rate'] - data_final['content_pronoun_rate']

## Save

In [ ]:
data_final.to_pickle('data_final_features.pkl')

print(f"Saved {len(data_final)} rows and {len(data_final.columns)} columns")
print("\nColumns saved:")
print(data_final.columns.tolist())